# Gold Layer - Aggregations & Reporting Tables

## Purpose 
Reads from the Silver Delta table and produces aggregated
Gold tables ready for reporting and dashboards.

## Reads from
- `pipeline_silver` — cleaned and transformed sales data

## Writes to
- `pipeline_gold_by_city` — revenue summary grouped by city
- `pipeline_gold_by_rep`  — revenue summary grouped by rep

## Notes
- No row-level transformations happen here
- All tables are aggregated and ready for Power BI
- Row counts are validated after each write

In [0]:
dbutils.widgets.text("run_date", "2024-01-01", "Run Date")
run_date = dbutils.widgets.get("run_date")

SOURCE_TABLE    = "pipeline_silver"
GOLD_CITY_TABLE = "pipeline_gold_by_city"
GOLD_REP_TABLE  = "pipeline_gold_by_rep"

print(f"Reading from: {SOURCE_TABLE}")
print(f"Run date:     {run_date}")

In [0]:
from pyspark.sql.functions import col, round, sum, avg, count, desc, rank
from pyspark.sql.window import Window
df_silver = spark.read.table(SOURCE_TABLE)

df_gold_by_city = df_silver.groupBy("city").agg(
    round(sum("amount"), 2).alias("total_revenue"),
    count("*").alias("total_orders"),
    round(avg("amount"), 2).alias("avg_order_value")
).orderBy(desc("total_revenue"))
spark.sql(f"DROP TABLE IF EXISTS {GOLD_CITY_TABLE}")
df_gold_by_city.write.mode("overwrite").format("delta").saveAsTable(GOLD_CITY_TABLE)
df_gold_by_city.show()

(GOLD_REP_TABLE)
df_gold_by_rep = df_silver.groupBy("rep","city").agg(
    round(sum("amount"), 2).alias("total_revenue"),
    count("*").alias("total_orders"),
    round(avg("amount"), 2).alias("avg_order_value")
).orderBy(desc("total_revenue"))
spark.sql(f"DROP TABLE IF EXISTS {GOLD_REP_TABLE}")
df_gold_by_rep.write.mode("overwrite").format("delta").saveAsTable(GOLD_REP_TABLE)
df_gold_by_rep.show()

In [0]:
city_count = spark.read.table(GOLD_CITY_TABLE).count()
rep_count  = spark.read.table(GOLD_REP_TABLE).count()

print(f"{GOLD_CITY_TABLE} — {city_count} rows")
print(f"{GOLD_REP_TABLE}  — {rep_count} rows")

if city_count == 0 or rep_count == 0:
    raise Exception("GOLD FAILED: One or more tables are empty")

print(f"Gold layer complete — run date: {run_date}")